In [3]:
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np

def design_and_optimize_section(b, D, d_dash, Mu, Vu, fck, fy):
    """
    Standard RCC Limit State Beam Design (IS 456 / Eurocode principles)
    b       : Width (mm)
    D       : Total depth (mm)
    d_dash  : Effective cover (mm)
    Mu      : Factored Bending Moment (kNm)
    Vu      : Factored Shear Force (kN)
    fck     : Concrete Characteristic Strength (MPa)
    fy      : Steel Yield Strength (MPa)
    """
    d = D - d_dash  # Effective depth
    
    # 1. Moment Capacity / Flexure Design
    # Limiting moment factor for Fe415 = 0.138, Fe500 = 0.133
    mu_factor = 0.138 if fy <= 415 else 0.133
    Mu_lim = (mu_factor * fck * b * (d ** 2)) / 1e6  # in kNm
    
    # Required tension steel (Ast)
    term = 1 - ((4.59 * (Mu * 1e6)) / (fck * b * (d ** 2)))
    
    if term < 0 or Mu > Mu_lim:
        section_status = "DOUBLY REINFORCED / REDESIGN (Section depth is insufficient for bending)"
        ast_req = 0.0
        pt_provided = 0.0
        is_safe_flexure = False
    else:
        ast_req = (0.5 * (fck / fy) * (1 - np.sqrt(term))) * b * d
        # Minimum steel requirement (0.85 * b * d / fy)
        ast_min = (0.85 * b * d) / fy
        ast_req = max(ast_req, ast_min)
        pt_provided = (ast_req / (b * d)) * 100
        is_safe_flexure = True
        
        if Mu / Mu_lim < 0.40:
            section_status = "OVER-DESIGNED (Consider reducing depth 'D' to save concrete)"
        else:
            section_status = "OPTIMAL (Singly Reinforced)"

    # 2. Shear Stress Check
    tau_v = (Vu * 1000.0) / (b * d)  # Nominal shear stress (MPa)
    # Permissible shear strength approximation (IS 456 Table 19)
    tau_c = 0.62 * np.sqrt(fck) * 0.5  # Base capacity without shear links
    tau_c_max = 0.62 * np.sqrt(fck)
    
    if tau_v > tau_c_max:
        shear_status = "SHEAR FAIL (Increase width 'b' or depth 'D')"
        is_safe_shear = False
    elif tau_v > tau_c:
        shear_status = "PASS (Requires Designed Shear Stirrups)"
        is_safe_shear = True
    else:
        shear_status = "PASS (Nominal Minimum Stirrups Required)"
        is_safe_shear = True

    # 3. Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
    
    # Plot Cross-Section
    ax1.set_xlim(-50, b + 50)
    ax1.set_ylim(-50, D + 50)
    ax1.add_patch(plt.Rectangle((0, 0), b, D, fill=True, color="#d9d9d9", ec="black", lw=2, label="Concrete Section"))
    
    # Draw Stirrup Outline
    ax1.add_patch(plt.Rectangle((25, 25), b - 50, D - 50, fill=False, edgecolor="red", linestyle="--", lw=1.5, label="Shear Stirrups"))
    
    # Draw Tension Rebars
    num_bars = max(2, int(np.ceil(ast_req / 314.0))) if is_safe_flexure else 2  # Assuming 20mm rebar (314 mm2)
    bar_x = np.linspace(40, b - 40, num_bars)
    ax1.scatter(bar_x, [d_dash] * num_bars, color="darkblue", s=120, zorder=5, label=f"{num_bars}x Main Bars (Tension)")
    
    ax1.set_title(f"RCC Section: {b:.0f} mm x {D:.0f} mm", fontsize=11, fontweight="bold")
    ax1.set_xlabel("Width (mm)")
    ax1.set_ylabel("Depth (mm)")
    ax1.legend(loc="upper right", fontsize=8)
    ax1.grid(True, linestyle=":", alpha=0.6)
    
    # Plot Demand vs Capacity Bar Chart
    bars = ["Bending Moment (kNm)", "Shear Stress (MPa)"]
    demands = [Mu, tau_v]
    capacities = [Mu_lim, tau_c_max]
    
    x = np.arange(len(bars))
    width = 0.3
    ax2.bar(x - width/2, demands, width, label='Applied Demand', color='#e74c3c')
    ax2.bar(x + width/2, capacities, width, label='Limit Capacity', color='#2ecc71')
    ax2.set_xticks(x)
    ax2.set_xticklabels(bars)
    ax2.set_ylabel("Magnitude")
    ax2.set_title("Demand vs. Section Capacity", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', linestyle=":", alpha=0.7)
    
    plt.tight_layout()
    plt.show()

    # 4. Formatted Summary Output
    print("-" * 60)
    print(f"Moment Demand (Mu)      : {Mu:.2f} kNm  | Capacity (Mu_lim): {Mu_lim:.2f} kNm")
    print(f"Shear Demand (tau_v)    : {tau_v:.3f} MPa | Max Capacity   : {tau_c_max:.3f} MPa")
    print(f"Required Ast (Tension)  : {ast_req:.1f} mm² (Pt = {pt_provided:.2f}%)")
    print(f"Flexure Check Status    : {section_status}")
    print(f"Shear Check Status      : {shear_status}")
    print("-" * 60)

# Set up Interactive Sliders
widgets.interact(
    design_and_optimize_section,
    b=widgets.IntSlider(min=200, max=600, step=25, value=250, description='Width b (mm)'),
    D=widgets.IntSlider(min=300, max=1000, step=25, value=500, description='Depth D (mm)'),
    d_dash=widgets.IntSlider(min=25, max=60, step=5, value=40, description='Cover (mm)'),
    Mu=widgets.FloatSlider(min=10.0, max=500.0, step=5.0, value=120.0, description='Moment Mu (kNm)'),
    Vu=widgets.FloatSlider(min=10.0, max=300.0, step=5.0, value=80.0, description='Shear Vu (kN)'),
    fck=widgets.Dropdown(options=[20, 25, 30, 35, 40], value=25, description='fck (MPa)'),
    fy=widgets.Dropdown(options=[415, 500, 550], value=415, description='fy (MPa)')
);

interactive(children=(IntSlider(value=250, description='Width b (mm)', max=600, min=200, step=25), IntSlider(v…

In [3]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

def sustainability_analysis(b, D, length_m, Mu, fck, fy, carbon_tax_per_ton):
    # Effective depth
    d = D - 40
    
    # Structural capacity check (IS 456 / Eurocode flexure limit)
    mu_lim = (0.138 * fck * b * (d ** 2)) / 1e6  # kNm
    term = 1 - ((4.59 * (Mu * 1e6)) / (fck * b * (d ** 2)))
    
    if term < 0 or Mu > mu_lim:
        print("STRUCTURAL STATUS: Section Failing / Requires Redesign.")
        return
    
    # Calculate Reinforcement Required
    ast = max((0.5 * (fck / fy) * (1 - np.sqrt(term))) * b * d, (0.85 * b * d) / fy)
    
    # Volume and Mass Calculations
    vol_concrete = (b / 1000.0) * (D / 1000.0) * length_m  # m3
    mass_concrete = vol_concrete * 2400.0                   # kg (density = 2400 kg/m3)
    
    vol_steel = (ast / 1e6) * length_m                     # m3
    mass_steel = vol_steel * 7850.0                        # kg (density = 7850 kg/m3)
    
    # Embodied Carbon Coefficients (ICE Database standard averages)
    # Concrete C25/30 ~ 0.13 kgCO2e/kg, Structural Rebar ~ 1.74 kgCO2e/kg
    co2_concrete = mass_concrete * (0.10 + 0.0012 * fck)
    co2_steel = mass_steel * 1.74
    total_co2 = co2_concrete + co2_steel
    
    # Direct Material Cost Estimates
    cost_concrete = vol_concrete * 120.0  # $120/m3
    cost_steel = mass_steel * 1.10        # $1.10/kg
    carbon_tax = (total_co2 / 1000.0) * carbon_tax_per_ton
    total_cost = cost_concrete + cost_steel + carbon_tax

    # Visualizations
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    # Carbon Breakdown
    ax1.pie([co2_concrete, co2_steel], labels=['Concrete', 'Rebar Steel'], 
            autopct='%1.1f%%', colors=['#7f8c8d', '#e67e22'], startangle=140, explode=(0.05, 0.05))
    ax1.set_title(f"Embodied Carbon: {total_co2:.1f} kgCO₂e", fontweight='bold')
    
    # Cost Breakdown
    categories = ['Concrete', 'Steel', 'Carbon Tax']
    costs = [cost_concrete, cost_steel, carbon_tax]
    ax2.bar(categories, costs, color=['#34495e', '#d35400', '#27ae60'])
    ax2.set_ylabel("Cost ($ USD)")
    ax2.set_title(f"Total Structural Cost: ${total_cost:.2f}", fontweight='bold')
    ax2.grid(axis='y', linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    plt.show()

widgets.interact(
    sustainability_analysis,
    b=widgets.IntSlider(min=200, max=600, step=25, value=300, description='Width b (mm)'),
    D=widgets.IntSlider(min=300, max=900, step=25, value=500, description='Depth D (mm)'),
    length_m=widgets.FloatSlider(min=3.0, max=10.0, step=0.5, value=6.0, description='Span (m)'),
    Mu=widgets.FloatSlider(min=20.0, max=350.0, step=5.0, value=140.0, description='Moment (kNm)'),
    fck=widgets.Dropdown(options=[20, 25, 30, 35, 40], value=25, description='fck (MPa)'),
    fy=widgets.Dropdown(options=[415, 500], value=415, description='fy (MPa)'),
    carbon_tax_per_ton=widgets.IntSlider(min=0, max=150, step=10, value=50, description='CO₂ Tax ($/ton)')
);

interactive(children=(IntSlider(value=300, description='Width b (mm)', max=600, min=200, step=25), IntSlider(v…

In [1]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

def multi_code_check(code_choice, b, D, Mu, fck, fy):
    d = D - 40
    
    if code_choice == 'ACI 318-19':
        # ACI 318-19 limit state using beta1 and phi = 0.90
        beta1 = max(0.65, 0.85 - (0.05 * (fck - 28) / 7)) if fck > 28 else 0.85
        phi = 0.90
        # Nominal moment capacity approximation
        term = 1 - (2 * (Mu * 1e6) / (phi * 0.85 * fck * b * (d ** 2)))
        if term < 0:
            status, ast, cap = "FAIL (Compression Failure)", 0, 0
        else:
            a = d * (1 - np.sqrt(term))
            ast = (0.85 * fck * a * b) / fy
            cap = (phi * ast * fy * (d - a / 2)) / 1e6

    elif code_choice == 'Eurocode 2 (EN 1992)':
        # EC2: gamma_c = 1.5, gamma_s = 1.15, alpha_cc = 0.85
        fcd = 0.85 * fck / 1.5
        fyd = fy / 1.15
        mu_dimless = (Mu * 1e6) / (b * (d ** 2) * fcd)
        mu_lim = 0.167
        if mu_dimless > mu_lim:
            status, ast, cap = "FAIL (Exceeds Limiting K)", 0, 0
        else:
            kz = 0.5 * (1 + np.sqrt(1 - 3.53 * mu_dimless))
            z = min(0.95 * d, kz * d)
            ast = (Mu * 1e6) / (fyd * z)
            cap = (ast * fyd * z) / 1e6

    else:  # IS 456:2000
        mu_factor = 0.138 if fy == 415 else 0.133
        mu_lim = (mu_factor * fck * b * (d ** 2)) / 1e6
        term = 1 - ((4.59 * (Mu * 1e6)) / (fck * b * (d ** 2)))
        if term < 0 or Mu > mu_lim:
            status, ast, cap = "FAIL (Over-reinforced)", 0, mu_lim
        else:
            ast = (0.5 * (fck / fy) * (1 - np.sqrt(term))) * b * d
            cap = mu_lim

    # Output Visualizer
    print("=" * 60)
    print(f"STANDARD CODE APPLIED : {code_choice}")
    print(f"Bending Demand (Mu)   : {Mu:.2f} kNm")
    print(f"Calculated Capacity   : {cap:.2f} kNm")
    print(f"Steel Required (Ast)  : {ast:.1f} mm²")
    print(f"Status                : {'SAFE / PASS' if ast > 0 else 'FAIL / REDESIGN'}")
    print("=" * 60)

widgets.interact(
    multi_code_check,
    code_choice=widgets.Dropdown(options=['ACI 318-19', 'Eurocode 2 (EN 1992)', 'IS 456:2000'], value='ACI 318-19', description='Standard:'),
    b=widgets.IntSlider(min=200, max=600, step=25, value=300, description='Width (mm)'),
    D=widgets.IntSlider(min=300, max=800, step=25, value=500, description='Depth (mm)'),
    Mu=widgets.FloatSlider(min=20.0, max=400.0, step=10.0, value=160.0, description='Demand (kNm)'),
    fck=widgets.Dropdown(options=[25, 30, 35, 40], value=30, description='fck (MPa)'),
    fy=widgets.Dropdown(options=[415, 500], value=500, description='fy (MPa)'),
    # Example snippet to compare Ast across standards visually
    codes = ['ACI 318-19', 'Eurocode 2', 'IS 456:2000']
    # Plotting a bar chart of Ast across codes gives an immediate literature review insight
);

interactive(children=(Dropdown(description='Standard:', options=('ACI 318-19', 'Eurocode 2 (EN 1992)', 'IS 456…

In [2]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

def detailed_rebar_section(b, D, cover, num_bars, bar_dia, stirrup_dia, agg_size):
    # Constructability Check: Clear Horizontal Distance
    total_cover_both_sides = 2 * (cover + stirrup_dia)
    usable_width = b - total_cover_both_sides
    total_bar_space = num_bars * bar_dia
    
    if num_bars > 1:
        clear_spacing = (usable_width - total_bar_space) / (num_bars - 1)
    else:
        clear_spacing = usable_width

    # Minimum allowed clear distance per standard code rules
    min_allowed_spacing = max(20.0, agg_size + 5.0, bar_dia)
    spacing_safe = clear_spacing >= min_allowed_spacing

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-40, b + 40)
    ax.set_ylim(-40, D + 40)
    
    # 1. Concrete Member
    ax.add_patch(plt.Rectangle((0, 0), b, D, color="#e0e0e0", ec="#2c3e50", lw=2.5, label="Concrete Section"))
    
    # 2. Shear Stirrup Link
    stirrup_x = cover
    stirrup_y = cover
    stirrup_w = b - 2 * cover
    stirrup_h = D - 2 * cover
    ax.add_patch(plt.Rectangle((stirrup_x, stirrup_y), stirrup_w, stirrup_h, 
                               fill=False, edgecolor="#c0392b", linestyle="-", lw=2, label=f"Stirrup Link (Ø{stirrup_dia}mm)"))
    
    # 3. Top Hanger Bars (2x12mm anchor bars)
    hanger_y = D - cover - stirrup_dia - 6
    ax.scatter([cover + stirrup_dia + 6, b - (cover + stirrup_dia + 6)], [hanger_y, hanger_y], 
               color="#2980b9", s=100, zorder=5, label="2x Top Hangers (Ø12mm)")
    
    # 4. Main Tension Reinforcement
    start_x = cover + stirrup_dia + (bar_dia / 2.0)
    end_x = b - cover - stirrup_dia - (bar_dia / 2.0)
    x_positions = np.linspace(start_x, end_x, num_bars)
    tension_y = cover + stirrup_dia + (bar_dia / 2.0)
    
    bar_color = "#27ae60" if spacing_safe else "#e74c3c"
    ax.scatter(x_positions, [tension_y] * num_bars, color=bar_color, s=(bar_dia ** 1.8) * 3, zorder=5, 
               label=f"{num_bars}x Main Rebar (Ø{bar_dia}mm)")

    ax.set_title(f"Rebar Detailing: {b:.0f}x{D:.0f} mm\nClear Spacing: {clear_spacing:.1f} mm (Min: {min_allowed_spacing:.1f} mm)", 
                 fontweight="bold", color="green" if spacing_safe else "red")
    ax.set_xlabel("Width (mm)")
    ax.set_ylabel("Depth (mm)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, linestyle=":", alpha=0.5)
    
    plt.tight_layout()
    plt.show()

widgets.interact(
    detailed_rebar_section,
    b=widgets.IntSlider(min=200, max=500, step=25, value=250, description='Width (mm)'),
    D=widgets.IntSlider(min=300, max=800, step=25, value=500, description='Depth (mm)'),
    cover=widgets.IntSlider(min=25, max=50, step=5, value=30, description='Cover (mm)'),
    num_bars=widgets.IntSlider(min=2, max=6, step=1, value=4, description='No. of Bars'),
    bar_dia=widgets.Dropdown(options=[12, 16, 20, 25, 32], value=20, description='Bar Dia (mm)'),
    stirrup_dia=widgets.Dropdown(options=[8, 10, 12], value=8, description='Stirrup Dia'),
    agg_size=widgets.Dropdown(options=[10, 20, 25], value=20, description='Max Agg (mm)')
);

interactive(children=(IntSlider(value=250, description='Width (mm)', max=500, min=200, step=25), IntSlider(val…

In [6]:
import ipywidgets as widgets
import pandas as pd
import numpy as np

def run_generative_optimizer(Mu_applied, Vu_applied, fck, fy):
    candidate_sections = []
    
    # Parametric search space
    widths = range(200, 501, 50)     # 200mm to 500mm
    depths = range(300, 851, 50)     # 300mm to 850mm
    
    for b in widths:
        for D in depths:
            d = D - 40
            
            # Flexure Capacity Limit Check
            mu_factor = 0.138 if fy == 415 else 0.133
            Mu_lim = (mu_factor * fck * b * (d ** 2)) / 1e6
            
            if Mu_applied > Mu_lim:
                continue  # Exclude under-designed sections
                
            term = 1 - ((4.59 * (Mu_applied * 1e6)) / (fck * b * (d ** 2)))
            if term < 0:
                continue
                
            ast = max((0.5 * (fck / fy) * (1 - np.sqrt(term))) * b * d, (0.85 * b * d) / fy)
            pt = (ast / (b * d)) * 100
            
            # Shear Stress Check
            tau_v = (Vu_applied * 1000.0) / (b * d)
            tau_c_max = 0.62 * np.sqrt(fck)
            if tau_v > tau_c_max:
                continue
                
            # Cost Function (Material Volume Cost per meter length)
            vol_conc = (b / 1000.0) * (D / 1000.0) * 1.0
            mass_steel = (ast / 1e6) * 1.0 * 7850.0
            cost_index = (vol_conc * 120.0) + (mass_steel * 1.10)
            
            candidate_sections.append({
                "Width (mm)": b,
                "Depth (mm)": D,
                "Ast (mm²)": round(ast, 1),
                "Pt (%)": round(pt, 2),
                "Capacity (kNm)": round(Mu_lim, 1),
                "D/C Ratio": round(Mu_applied / Mu_lim, 2),
                "Cost Index ($/m)": round(cost_index, 2)
            })
            
    df = pd.DataFrame(candidate_sections)
    if df.empty:
        print("No valid sections found for the applied load. Increase concrete strength or member bounds.")
        return
        
    df_sorted = df.sort_values(by="Cost Index ($/m)").reset_index(drop=True)
    
    print("=" * 70)
    print("        OPTIMIZATION COMPLETE: TOP 3 MOST ECONOMICAL SECTIONS")
    print("=" * 70)
    display(df_sorted.head(3))
    print("=" * 70)

widgets.interact_manual(
    run_generative_optimizer,
    Mu_applied=widgets.FloatSlider(min=50.0, max=400.0, step=10.0, value=150.0, description='Moment (kNm)'),
    Vu_applied=widgets.FloatSlider(min=20.0, max=250.0, step=10.0, value=75.0, description='Shear (kN)'),
    fck=widgets.Dropdown(options=[25, 30, 35, 40], value=25, description='fck (MPa)'),
    fy=widgets.Dropdown(options=[415, 500], value=415, description='fy (MPa)')
);

interactive(children=(FloatSlider(value=150.0, description='Moment (kNm)', max=400.0, min=50.0, step=10.0), Fl…

In [7]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

def comprehensive_rcc_design(b, D, cover, span_m, Mu, Vu, fck, fy):
    d = D - cover
    d_dash = cover  # Depth to compression steel
    
    # -------------------------------------------------------------
    # 1. FLEXURE DESIGN (Singly vs. Doubly Reinforced Logic)
    # -------------------------------------------------------------
    mu_factor = 0.138 if fy <= 415 else 0.133
    Mu_lim = (mu_factor * fck * b * (d ** 2)) / 1e6  # Limiting capacity (kNm)
    
    if Mu <= Mu_lim:
        # Singly Reinforced Section
        term = 1 - ((4.59 * (Mu * 1e6)) / (fck * b * (d ** 2)))
        Ast1 = (0.5 * (fck / fy) * (1 - np.sqrt(max(0, term)))) * b * d
        Ast_min = (0.85 * b * d) / fy
        Ast_total = max(Ast1, Ast_min)
        Asc = 0.0
        design_type = "SINGLY REINFORCED"
    else:
        # Doubly Reinforced Section (Mu > Mu_lim)
        Mu2 = Mu - Mu_lim  # Extra moment to be carried by compression steel
        # Stress in compression steel (fsc approx for Fe415 ~ 355 MPa, Fe500 ~ 412 MPa)
        fsc = 0.85 * fy if (d_dash / d) <= 0.1 else 0.80 * fy
        fcd = 0.446 * fck
        
        # Steel calculations
        Ast1 = (0.5 * (fck / fy) * (1 - np.sqrt(1 - (4.59 * (Mu_lim * 1e6) / (fck * b * (d ** 2)))))) * b * d
        Asc = (Mu2 * 1e6) / ((fsc - fcd) * (d - d_dash))
        Ast2 = (Asc * fsc) / (0.87 * fy)
        Ast_total = Ast1 + Ast2
        design_type = "DOUBLY REINFORCED (Heavy Moment)"

    pt = (Ast_total / (b * d)) * 100

    # -------------------------------------------------------------
    # 2. SERVICEABILITY LIMIT STATE (SLS Deflection / L/d Check)
    # -------------------------------------------------------------
    actual_L_over_d = (span_m * 1000.0) / d
    # Basic L/d ratio for simply supported beam = 20
    # Modification factor (approximate based on tension steel percentage pt)
    fs = 0.58 * fy * (Ast1 / Ast_total) if Ast_total > 0 else 240.0
    mod_factor = min(2.0, max(0.8, 1.0 / (0.225 + 0.0032 * fs - 0.625 * np.log10(max(pt, 0.1)))))
    allowable_L_over_d = 20.0 * mod_factor
    
    deflection_safe = actual_L_over_d <= allowable_L_over_d

    # -------------------------------------------------------------
    # 3. SHEAR CHECK
    # -------------------------------------------------------------
    tau_v = (Vu * 1000.0) / (b * d)
    tau_c_max = 0.62 * np.sqrt(fck)
    shear_safe = tau_v <= tau_c_max

    # -------------------------------------------------------------
    # 4. VISUALIZATION
    # -------------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    
    # Cross section drawing
    ax1.set_xlim(-40, b + 40)
    ax1.set_ylim(-40, D + 40)
    ax1.add_patch(plt.Rectangle((0, 0), b, D, color="#e5e7eb", ec="#111827", lw=2, label="Concrete Section"))
    ax1.add_patch(plt.Rectangle((cover, cover), b - 2*cover, D - 2*cover, fill=False, ec="red", ls="--", lw=1.5, label="Stirrup Link"))
    
    # Tension bars (Bottom)
    num_tension_bars = max(2, int(np.ceil(Ast_total / 314.0)))  # assuming 20mm bars
    x_bot = np.linspace(cover + 15, b - cover - 15, num_tension_bars)
    ax1.scatter(x_bot, [cover + 10] * num_tension_bars, color="#1e40af", s=120, zorder=5, label=f"Tension: {num_tension_bars}xØ20mm")
    
    # Compression bars (Top)
    if Asc > 0:
        num_comp_bars = max(2, int(np.ceil(Asc / 201.0)))  # assuming 16mm bars
        x_top = np.linspace(cover + 15, b - cover - 15, num_comp_bars)
        ax1.scatter(x_top, [D - cover - 10] * num_comp_bars, color="#b91c1c", s=100, zorder=5, label=f"Comp (Asc): {num_comp_bars}xØ16mm")
    else:
        # Standard hanger bars
        ax1.scatter([cover + 15, b - cover - 15], [D - cover - 10, D - cover - 10], color="#6b7280", s=60, zorder=5, label="2x Top Hangers")
        
    ax1.set_title(f"Section ({design_type})", fontweight="bold", fontsize=10)
    ax1.legend(loc="upper right", fontsize=8)
    ax1.grid(True, linestyle=":", alpha=0.5)

    # Deflection (L/d) & Flexure Comparison Chart
    metrics = ['Span/Depth Ratio (L/d)', 'Moment (kNm)']
    actual_vals = [actual_L_over_d, Mu]
    limit_vals = [allowable_L_over_d, Mu_lim]
    
    x = np.arange(len(metrics))
    width = 0.3
    ax2.bar(x - width/2, actual_vals, width, label='Applied / Actual', color='#ef4444')
    ax2.bar(x + width/2, limit_vals, width, label='Limiting / Allowable', color='#22c55e')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metrics)
    ax2.set_title("ULS & SLS Limits Check", fontweight="bold", fontsize=10)
    ax2.legend(fontsize=8)
    ax2.grid(axis='y', linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 5. CONSOLE REPORT
    # -------------------------------------------------------------
    print("=" * 65)
    print(f"DESIGN CLASSIFICATION    : {design_type}")
    print(f"Applied Moment (Mu)      : {Mu:.2f} kNm  | Limiting (Mu_lim): {Mu_lim:.2f} kNm")
    print(f"Tension Steel (Ast)      : {Ast_total:.1f} mm² (Pt = {pt:.2f}%)")
    print(f"Compression Steel (Asc)  : {Asc:.1f} mm²")
    print(f"Deflection (L/d) Check   : Actual = {actual_L_over_d:.1f} | Allowable = {allowable_L_over_d:.1f} -> {'PASS' if deflection_safe else 'FAIL (Deflection Exceeded)'}")
    print(f"Shear Stress Check       : tau_v = {tau_v:.2f} MPa -> {'PASS' if shear_safe else 'FAIL (Increase Section)'}")
    print("=" * 65)

widgets.interact(
    comprehensive_rcc_design,
    b=widgets.IntSlider(min=200, max=500, step=25, value=250, description='Width (mm)'),
    D=widgets.IntSlider(min=300, max=800, step=25, value=450, description='Depth (mm)'),
    cover=widgets.IntSlider(min=25, max=50, step=5, value=35, description='Cover (mm)'),
    span_m=widgets.FloatSlider(min=3.0, max=10.0, step=0.5, value=5.5, description='Span (m)'),
    Mu=widgets.FloatSlider(min=50.0, max=400.0, step=10.0, value=170.0, description='Moment (kNm)'),
    Vu=widgets.FloatSlider(min=20.0, max=250.0, step=5.0, value=75.0, description='Shear (kN)'),
    fck=widgets.Dropdown(options=[20, 25, 30, 35, 40], value=25, description='fck (MPa)'),
    fy=widgets.Dropdown(options=[415, 500], value=415, description='fy (MPa)')
);

interactive(children=(IntSlider(value=250, description='Width (mm)', max=500, min=200, step=25), IntSlider(val…

In [9]:
import json
import os
from datetime import datetime

def generate_bim_and_calc_report(b, D, cover, span_m, Mu, Vu, fck, fy, selected_code="IS 456:2000"):
    """
    Automated Structural Calculation Sheet & BIM Integration Payload Generator.
    Exports structural design attributes to JSON (for Revit/Dynamo) and formats a text-based calc sheet.
    """
    d = D - cover
    d_dash = cover
    
    # 1. Structural Calculation Logic
    mu_factor = 0.138 if fy <= 415 else 0.133
    Mu_lim = (mu_factor * fck * b * (d ** 2)) / 1e6  # kNm
    
    if Mu <= Mu_lim:
        term = 1 - ((4.59 * (Mu * 1e6)) / (fck * b * (d ** 2)))
        Ast1 = (0.5 * (fck / fy) * (1 - np.sqrt(max(0, term)))) * b * d
        Ast_min = (0.85 * b * d) / fy
        Ast_total = max(Ast1, Ast_min)
        Asc = 0.0
        design_type = "SINGLY REINFORCED"
    else:
        Mu2 = Mu - Mu_lim
        fsc = 0.85 * fy if (d_dash / d) <= 0.1 else 0.80 * fy
        fcd = 0.446 * fck
        Ast1 = (0.5 * (fck / fy) * (1 - np.sqrt(1 - (4.59 * (Mu_lim * 1e6) / (fck * b * (d ** 2)))))) * b * d
        Asc = (Mu2 * 1e6) / ((fsc - fcd) * (d - d_dash))
        Ast2 = (Asc * fsc) / (0.87 * fy)
        Ast_total = Ast1 + Ast2
        design_type = "DOUBLY REINFORCED"

    # Rebar layout estimation
    num_tension_bars = max(2, int(np.ceil(Ast_total / 314.0)))  # 20mm rebar base
    num_comp_bars = max(2, int(np.ceil(Asc / 201.0))) if Asc > 0 else 2  # 16mm rebar base
    
    # Serviceability Check
    pt = (Ast_total / (b * d)) * 100
    actual_L_d = (span_m * 1000.0) / d
    fs = 0.58 * fy * (Ast1 / Ast_total) if Ast_total > 0 else 240.0
    mod_factor = min(2.0, max(0.8, 1.0 / (0.225 + 0.0032 * fs - 0.625 * np.log10(max(pt, 0.1)))))
    allowable_L_d = 20.0 * mod_factor
    deflection_status = "PASS" if actual_L_d <= allowable_L_d else "FAIL"

    # 2. Construct BIM JSON Payload (For Revit Dynamo / API)
    bim_payload = {
        "metadata": {
            "project_name": "Computational RCC Automation Suite",
            "export_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "design_code": selected_code
        },
        "member_geometry": {
            "element_type": "Structural Framing / RCC Beam",
            "width_b_mm": b,
            "depth_D_mm": D,
            "effective_depth_d_mm": d,
            "clear_cover_mm": cover,
            "span_length_m": span_m
        },
        "material_properties": {
            "fck_concrete_MPa": fck,
            "fy_steel_MPa": fy
        },
        "loading_demands": {
            "factored_moment_Mu_kNm": Mu,
            "factored_shear_Vu_kN": Vu
        },
        "reinforcement_results": {
            "design_classification": design_type,
            "required_Ast_mm2": round(Ast_total, 2),
            "required_Asc_mm2": round(Asc, 2),
            "tension_bars": {"count": num_tension_bars, "diameter_mm": 20},
            "compression_bars": {"count": num_comp_bars, "diameter_mm": 16},
            "reinforcement_percentage_pt": round(pt, 2)
        },
        "serviceability_and_compliance": {
            "actual_L_over_d": round(actual_L_d, 2),
            "allowable_L_over_d": round(allowable_L_d, 2),
            "deflection_status": deflection_status,
            "overall_status": "APPROVED" if deflection_status == "PASS" else "REQUIRES_REVISION"
        }
    }

    # Save BIM JSON File
    json_filename = "rcc_beam_design_bim_payload.json"
    with open(json_filename, "w") as f:
        json.dump(bim_payload, f, indent=4)

    # 3. Formatted Structural Calculation Sheet Display
    calc_sheet = f"""
================================================================================
              STRUCTURAL CALCULATION SHEET & COMPLIANCE REPORT                  
================================================================================
Design Code Applied : {selected_code}
Timestamp           : {bim_payload['metadata']['export_timestamp']}
Element Type        : Concrete Beam Section

--- 1. SECTION GEOMETRY & MATERIALS ---
Width (b)           : {b} mm
Total Depth (D)     : {D} mm
Effective Depth (d) : {d} mm  (Cover = {cover} mm)
Span Length (L)     : {span_m} m
Concrete Strength   : C{fck} ({fck} MPa)
Rebar Yield Strength: Fe{fy} ({fy} MPa)

--- 2. FLEXURAL & SERVICEABILITY ANALYSIS ---
Applied Moment (Mu) : {Mu:.2f} kNm
Limiting Capacity   : {Mu_lim:.2f} kNm
Design Status       : {design_type}
Req. Tension Steel  : {Ast_total:.1f} mm² ({num_tension_bars}x Ø20mm)
Req. Comp. Steel    : {Asc:.1f} mm² ({num_comp_bars}x Ø16mm)
Span/Depth (L/d)    : Actual = {actual_L_d:.1f} | Allowable = {allowable_L_d:.1f} -> [{deflection_status}]

--- 3. BIM & AUTOMATION EXPORT ---
JSON BIM Payload    : Successfully written to '{json_filename}'
Revit Import Ready  : YES (Schema structured for Dynamo / Revit Python Script)
================================================================================
    """
    print(calc_sheet)

# Interactive Widget Trigger
widgets.interact_manual(
    generate_bim_and_calc_report,
    b=widgets.IntSlider(min=200, max=500, step=25, value=250, description='Width (mm)'),
    D=widgets.IntSlider(min=300, max=800, step=25, value=450, description='Depth (mm)'),
    cover=widgets.IntSlider(min=25, max=50, step=5, value=35, description='Cover (mm)'),
    span_m=widgets.FloatSlider(min=3.0, max=10.0, step=0.5, value=5.5, description='Span (m)'),
    Mu=widgets.FloatSlider(min=50.0, max=400.0, step=10.0, value=170.0, description='Moment (kNm)'),
    Vu=widgets.FloatSlider(min=20.0, max=250.0, step=5.0, value=75.0, description='Shear (kN)'),
    fck=widgets.Dropdown(options=[20, 25, 30, 35, 40], value=25, description='fck (MPa)'),
    fy=widgets.Dropdown(options=[415, 500], value=415, description='fy (MPa)'),
    selected_code=widgets.Dropdown(options=['IS 456:2000', 'ACI 318-19', 'Eurocode 2'], value='IS 456:2000', description='Code:')
);

interactive(children=(IntSlider(value=250, description='Width (mm)', max=500, min=200, step=25), IntSlider(val…